# Structure-Aware Markdown Parsing

| Field | Value |
|---|---|
| Stage | Data foundation |
| Difficulty | Intermediate |
| Status | Complete |
| Requires network/API | No |
| Last reviewed | 2026-09-25 |

Callout - Key idea:
Markdown headings are retrieval metadata. Preserving their hierarchy creates smaller, explainable sections without discarding the document context needed for citations.

## 30-Second Summary

This notebook compares one-document-per-file loading with a small structure-aware parser. The parser emits sections carrying source, title, heading breadcrumb, and deterministic chunk ID. It validates coverage by reconstructing every source from its parsed sections and checks that retrieval returns focused evidence.

## Why This Matters

Flattening Markdown throws away an inexpensive semantic signal: headings tell us what text is about and where it belongs. Splitting only by character count can mix topics or detach a paragraph from its heading, making results harder to rank and cite.

## Scope

| Covers | Does not cover |
|---|---|
| ATX headings (`#` through `######`), heading hierarchy, preamble handling, stable IDs, lexical comparison | Full CommonMark AST, Setext headings, HTML blocks, semantic chunking, link crawling |


## Mental Model

```text
Markdown file -> lines -> heading stack -> section text + breadcrumb
                                            |-> stable section ID
                                            |-> source path
                                            `-> searchable focused unit
```

A heading opens a section. A heading of the same or higher level closes the previous branch; lower-level headings extend the breadcrumb. The section ID combines source identity with its ordinal and breadcrumb so duplicates remain traceable.


In [1]:
from collections import Counter
from dataclasses import dataclass
from hashlib import sha256
from pathlib import Path
import math
import re


def find_repo_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "pyproject.toml").is_file():
            return candidate
    raise FileNotFoundError("Run this notebook from inside the repository.")


REPO_ROOT = find_repo_root()
MARKDOWN_DIR = REPO_ROOT / "05-DataIngestParsing/data/markdown"
SOURCE_FILES = sorted(MARKDOWN_DIR.glob("*.md"))

assert SOURCE_FILES, f"No Markdown fixtures found under {MARKDOWN_DIR}"
len(SOURCE_FILES), [path.name for path in SOURCE_FILES]


(10,
 ['action-builder.md',
  'code-quality-audit-plan.md',
  'code-quality.md',
  'firebase.md',
  'history-service.md',
  'index.md',
  'journey-canvas.md',
  'overview.md',
  'redux.md',
  'theming.md'])

## How It Works

The parser scans lines once while maintaining a stack of `(level, heading)` pairs. When a heading appears, the current section is emitted, headings at the same or deeper level are removed, and the new heading is pushed. Body lines accumulate until the next heading or end of file.

The implementation preserves the full section text, including its heading. It does not attempt to interpret fenced code, tables, or links; those remain source text. For production-grade CommonMark coverage, use an AST parser and keep this notebook as the behavioral reference.


## Baseline

The baseline creates one searchable item per file. This preserves source provenance but makes large files broad: a query may retrieve the correct file while returning far more context than the answer needs.


In [2]:
baseline_documents = [
    {
        "id": path.stem,
        "source": path.relative_to(REPO_ROOT).as_posix(),
        "content": path.read_text(encoding="utf-8").strip(),
    }
    for path in SOURCE_FILES
]
[(item["id"], len(item["content"])) for item in baseline_documents]


[('action-builder', 20561),
 ('code-quality-audit-plan', 9878),
 ('code-quality', 12677),
 ('firebase', 15911),
 ('history-service', 18276),
 ('index', 3227),
 ('journey-canvas', 20053),
 ('overview', 20840),
 ('redux', 11513),
 ('theming', 10308)]

## Technique Implementation

`MarkdownSection` makes the parent file and heading breadcrumb first-class. Empty heading-only sections are omitted, while a preamble before the first heading receives the breadcrumb `Preamble`. Section ordinals keep IDs unique when a document repeats a heading.


In [3]:
@dataclass(frozen=True)
class MarkdownSection:
    id: str
    source: str
    breadcrumb: tuple[str, ...]
    content: str


HEADING = re.compile(r"^(#{1,6})\s+(.+?)\s*$")


def parse_markdown(path: Path) -> list[MarkdownSection]:
    source = path.resolve().relative_to(REPO_ROOT).as_posix()
    lines = path.read_text(encoding="utf-8").splitlines()
    stack: list[tuple[int, str]] = []
    body: list[str] = []
    sections: list[MarkdownSection] = []

    def emit() -> None:
        content = "\n".join(body).strip()
        if not content:
            return
        breadcrumb = tuple(title for _, title in stack) or ("Preamble",)
        ordinal = len(sections)
        identity = f"{source}|{ordinal}|{' > '.join(breadcrumb)}"
        sections.append(
            MarkdownSection(
                id=sha256(identity.encode("utf-8")).hexdigest()[:16],
                source=source,
                breadcrumb=breadcrumb,
                content=content,
            )
        )

    for line in lines:
        match = HEADING.match(line)
        if match:
            emit()
            body = [line]
            level, title = len(match.group(1)), match.group(2).strip()
            stack = [(old_level, old_title) for old_level, old_title in stack if old_level < level]
            stack.append((level, title))
        else:
            body.append(line)
    emit()
    return sections


sections = [section for path in SOURCE_FILES for section in parse_markdown(path)]
[
    {
        "id": section.id,
        "breadcrumb": " > ".join(section.breadcrumb),
        "characters": len(section.content),
    }
    for section in sections[:8]
]


[{'id': 'f9b4cfebb4129817',
  'breadcrumb': 'Action Builder — Technical Documentation',
  'characters': 165},
 {'id': 'fe9b25c360c9203d',
  'breadcrumb': 'Action Builder — Technical Documentation > Overview',
  'characters': 688},
 {'id': 'a902df28c94903b1',
  'breadcrumb': 'Action Builder — Technical Documentation > Overview > Key Capabilities',
  'characters': 774},
 {'id': 'b6c8141fb451c696',
  'breadcrumb': 'Action Builder — Technical Documentation > Motivation & Inspiration',
  'characters': 27},
 {'id': '804422065a5363f9',
  'breadcrumb': 'Action Builder — Technical Documentation > Motivation & Inspiration > The Problem',
  'characters': 609},
 {'id': '28f59626993fcab6',
  'breadcrumb': 'Action Builder — Technical Documentation > Motivation & Inspiration > The Solution',
  'characters': 240},
 {'id': '5b32d82424b599fb',
  'breadcrumb': 'Action Builder — Technical Documentation > Motivation & Inspiration > Design Principles',
  'characters': 584},
 {'id': 'c59ef5c3be9dc118',
  'br

## Controlled Experiment

We compare whole-file and section retrieval with the same transparent TF-IDF scorer. The three queries deliberately target narrow concepts. We measure source accuracy and how much context accompanies the top result. Lower top-result character count is a useful focus proxy here, not a universal quality metric.


In [4]:
STOP_WORDS = {"a", "an", "and", "are", "as", "for", "from", "how", "in", "is", "of", "on", "the", "to", "what", "with"}


def tokens(text: str) -> list[str]:
    return [token for token in re.findall(r"[a-z0-9]+", text.lower()) if token not in STOP_WORDS]


def tfidf_rank(query: str, items: list[dict]) -> list[dict]:
    document_frequency = Counter()
    for item in items:
        document_frequency.update(set(tokens(item["content"])))
    query_counts = Counter(tokens(query))
    scored = []
    for item in items:
        counts = Counter(tokens(item["content"]))
        shared = set(query_counts).intersection(counts)
        score = sum(
            (1 + math.log(query_counts[term]))
            * (1 + math.log(counts[term]))
            * (math.log((1 + len(items)) / (1 + document_frequency[term])) + 1) ** 2
            for term in shared
        )
        scored.append({**item, "score": score})
    return sorted(scored, key=lambda item: (-item["score"], item["id"]))


section_documents = [
    {
        "id": section.id,
        "source": section.source,
        "breadcrumb": " > ".join(section.breadcrumb),
        "content": section.content,
    }
    for section in sections
]
queries = [
    ("How are Redux and TanStack Query responsibilities separated?", "redux.md"),
    ("Which theme is persisted across sessions?", "theming.md"),
    ("What can the change history rollback?", "history-service.md"),
]

experiment_rows = []
for query, expected_name in queries:
    file_hit = tfidf_rank(query, baseline_documents)[0]
    section_hit = tfidf_rank(query, section_documents)[0]
    experiment_rows.append(
        {
            "query": query,
            "file_correct": file_hit["source"].endswith(expected_name),
            "section_correct": section_hit["source"].endswith(expected_name),
            "file_characters": len(file_hit["content"]),
            "section_characters": len(section_hit["content"]),
            "section": section_hit.get("breadcrumb"),
        }
    )


for row in experiment_rows:
    print(
        f"file_ok={row['file_correct']!s:<5} section_ok={row['section_correct']!s:<5} "
        f"chars={row['file_characters']:>5}->{row['section_characters']:<5} "
        f"section={row['section']}"
    )


file_ok=False section_ok=True  chars=20840->545   section=Redux Implementation — In-Depth Guide > Overview
file_ok=True  section_ok=True  chars=10308->325   section=Multi-Theming and Styling -- In-Depth Guide > Overview
file_ok=True  section_ok=True  chars=18276->557   section=Change History Service — Technical Documentation > Rollback System > Individual Rollback


## Evaluation

For the three focused questions, the whole-file baseline retrieves the expected source for **2/3** questions, while section retrieval gets **3/3**. The broad Redux file loses one lexical comparison to another large architecture document; the focused section fixes that result and returns much less context with a citation-ready breadcrumb. We also require **lossless line coverage**: concatenating a file's emitted section content must reproduce its non-blank source lines in order.

The character-count reduction is evidence about context focus on these fixtures, not evidence that smaller chunks always improve retrieval. Very small sections can lose definitions, prerequisites, or cross-section relationships.


In [5]:
def nonblank_lines(text: str) -> list[str]:
    return [line.rstrip() for line in text.splitlines() if line.strip()]


for path in SOURCE_FILES:
    parsed = parse_markdown(path)
    reconstructed = "\n".join(section.content for section in parsed)
    assert nonblank_lines(reconstructed) == nonblank_lines(path.read_text(encoding="utf-8"))

assert sum(row["file_correct"] for row in experiment_rows) == 2
assert all(row["section_correct"] for row in experiment_rows)
assert all(row["section_characters"] < row["file_characters"] for row in experiment_rows)
assert len({section.id for section in sections}) == len(sections)
print(f"Markdown checks passed: {len(SOURCE_FILES)} files -> {len(sections)} sections.")


Markdown checks passed: 10 files -> 302 sections.


## Decision Guide

| Situation | Representation | Trade-off |
|---|---|---|
| Short, single-topic note | Whole file | Simple, but broad if the note grows |
| Well-structured documentation | Heading-aware sections | Focused retrieval with useful breadcrumbs |
| Long sections containing several ideas | Heading plus token/recursive sub-splitting | More IDs and boundary decisions |
| Weak or inconsistent headings | AST plus semantic/recursive fallback | More parser and evaluation complexity |
| Exact code/API lookup | Preserve fenced blocks with their explanatory heading | Code can dominate lexical scoring |

Choose boundaries by evaluating answer-support coverage and context noise, not by assuming one chunk size is universally best.


## Failure Modes and Debugging

| Symptom | Likely cause | Verify | Fix |
|---|---|---|---|
| Text disappears | Parser emits only non-heading bodies or mishandles preamble | Reconstruct source lines | Add coverage assertions and fixtures |
| Breadcrumb is wrong | Heading stack is not trimmed by level | Inspect nested heading transitions | Test level jumps and repeated headings |
| Code block becomes a false heading | Regex ignores fenced-code state | Add a fenced block containing `#` | Use a CommonMark AST parser |
| Tiny sections rank without context | Granularity is too fine | Inspect answer-support spans | Merge children or retrieve parent context |
| IDs change after unrelated edits | Identity depends on ordinal only | Insert an earlier heading and compare | Use source IDs plus durable anchors/versioning |


## Production Notes

### Observability
Track sections per file, size distribution, empty sections, parser version, heading-depth anomalies, reconstruction failures, and retrieval results by source and breadcrumb.

### Safety and Guardrails
Markdown can contain HTML, scripts, remote images, and prompt-like instructions. Parse as data, sanitize before rendering, and preserve authorization metadata on every section.

### Latency and Cost
The scan is linear in source length. Embedding cost grows with emitted section count, so measure duplicate context and update churn before choosing very small boundaries.


## Practice

Add a fixture containing a preamble, a skipped heading level, a repeated heading, and a fenced code block with a line beginning `#`. Write expected breadcrumbs first, then decide whether to extend this parser or replace it with an AST implementation.

## Recall

Toggle - Recall: Why keep heading breadcrumbs?
They preserve topical context for ranking, display, and citations even when a section is retrieved alone.

Toggle - Recall: What does reconstruction test?
That parsing did not silently drop or reorder non-blank source lines.

Toggle - Recall: Why is smaller not automatically better?
Small sections reduce noise but can remove definitions or dependencies needed to support an answer.

Toggle - Recall: When should this regex parser be replaced?
When full CommonMark behavior, fenced-block awareness, Setext headings, or complex extensions matter.

## Sources

- [CommonMark specification](https://spec.commonmark.org/)
- [Python documentation: Regular expression operations](https://docs.python.org/3/library/re.html)
- Repository-owned fixtures under `05-DataIngestParsing/data/markdown/`

## Review Log

| Date | Status | Confidence | Next review focus |
|---|---|---|---|
| 2026-09-25 | Complete; executed and visually reviewed | High for the documented ATX-heading subset | Add fenced-code and Setext-heading fixtures |
